# Φάση Γ: Classification Models

**Υπεύθυνος:** ML Engineer

**Μοντέλα:**
1. Random Forest
2. SVM


In [1]:
# =====================================================================
# Φάση Γ: Classification Models (ΠΛΗΡΗΣ ΥΛΟΠΟΙΗΣΗ)
# =====================================================================

from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.ml.classification import RandomForestClassifier, LinearSVC, NaiveBayes, MultilayerPerceptronClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. Αρχικοποίηση SparkSession
print("Αρχικοποίηση SparkSession...")
spark = SparkSession.builder \
    .appName("Stroke_Modeling_Classification") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# 2. Φόρτωση δεδομένων από το Gold Layer
print("Φόρτωση δεδομένων Gold Layer...")
train_gold = spark.read.parquet("../data/train_gold.parquet")
test_gold = spark.read.parquet("../data/test_gold.parquet")

# Cast σε Double για σιγουριά (απαιτείται από κάποια μοντέλα όπως SVM/NB)
train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))

evaluator = BinaryClassificationEvaluator(labelCol="stroke", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# --- ΜΟΝΤΕΛΟ 1: Random Forest ---
print("\nΕκπαίδευση Random Forest...")
rf = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=100, maxDepth=5, seed=42)
rf_model = rf.fit(train_gold)
rf_preds = rf_model.transform(test_gold)

# --- ΜΟΝΤΕΛΟ 2: Support Vector Machine (SVM) ---
print("Εκπαίδευση SVM...")
svm = LinearSVC(featuresCol="features", labelCol="stroke", maxIter=100)
svm_model = svm.fit(train_gold)
svm_preds = svm_model.transform(test_gold)

# --- ΜΟΝΤΕΛΟ 3: Naive Bayes (Με Cross-Validation & Hyperparameter Tuning) ---
print("Εκπαίδευση Naive Bayes με 3-Fold Cross-Validation...")
nb = NaiveBayes(featuresCol="features", labelCol="stroke")

nb_paramGrid = (ParamGridBuilder()
             .addGrid(nb.smoothing, [0.0, 0.5, 1.0, 2.0])
             .build())

nb_cv = CrossValidator(estimator=nb, 
                       estimatorParamMaps=nb_paramGrid, 
                       evaluator=evaluator, 
                       numFolds=3)

nb_cv_model = nb_cv.fit(train_gold)
nb_best_model = nb_cv_model.bestModel
nb_preds = nb_best_model.transform(test_gold)
print(f"Βέλτιστο smoothing για Naive Bayes: {nb_best_model.getOrDefault('smoothing')}")

# --- ΜΟΝΤΕΛΟ 4: Neural Networks (Multilayer Perceptron) ---
print("Εκπαίδευση Neural Network (MLP)...")
# Υπολογισμός μεγέθους input layer με βάση το vector των features
input_size = len(train_gold.select("features").first()[0])
layers = [input_size, 16, 8, 2] # Input -> Hidden(16) -> Hidden(8) -> Output(2)

mlp = MultilayerPerceptronClassifier(maxIter=100, layers=layers, blockSize=128, seed=42, 
                                     featuresCol="features", labelCol="stroke")
mlp_model = mlp.fit(train_gold)
mlp_preds = mlp_model.transform(test_gold)

# --- ΑΠΟΘΗΚΕΥΣΗ ΠΡΟΒΛΕΨΕΩΝ ---
print("\nΑποθήκευση προβλέψεων στο δίσκο...")
columns_to_save = ["stroke", "prediction", "probability"] # probability δεν υπάρχει στο LinearSVC

rf_preds.select(columns_to_save).write.mode("overwrite").parquet("../data/rf_predictions.parquet")
svm_preds.select("stroke", "prediction").write.mode("overwrite").parquet("../data/svm_predictions.parquet")
nb_preds.select(columns_to_save).write.mode("overwrite").parquet("../data/nb_predictions.parquet")
mlp_preds.select(columns_to_save).write.mode("overwrite").parquet("../data/mlp_predictions.parquet")

print("Όλα τα Classification Models εκπαιδεύτηκαν και αποθηκεύτηκαν!")

Αρχικοποίηση SparkSession...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/03 18:23:57 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp4s0)
26/06/03 18:23:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 18:23:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Φόρτωση δεδομένων Gold Layer...

Εκπαίδευση Random Forest...
Εκπαίδευση SVM...
Εκπαίδευση Naive Bayes με 3-Fold Cross-Validation...
Βέλτιστο smoothing για Naive Bayes: 0.0
Εκπαίδευση Neural Network (MLP)...

Αποθήκευση προβλέψεων στο δίσκο...
Όλα τα Classification Models εκπαιδεύτηκαν και αποθηκεύτηκαν!
